# Intro to Raster data

## Raster vs Vector Data in GIS

### Raster Data — Strengths

- ✓ Efficient for grid-based computations (matrix operations, map algebra)
- ✓ Ideal for continuous surfaces such as:
  - Elevation
  - Temperature
  - Rainfall
  - Air quality / pollution
- ✓ Well-suited for satellite, drone, and remote sensing imagery
- ✓ Powerful for spatial modeling, simulation, and machine learning workflows
- ✓ Naturally represents spatial variation across space

---

### Vector Data — Strengths

- ✓ Efficient representation of discrete geographic features such as:
  - Roads
  - Buildings
  - Parcels
  - Administrative boundaries
- ✓ Compact storage for sparse spatial features
- ✓ High geometric precision (points, lines, polygons)
- ✓ Strong attribute support (tables linked to spatial features)
- ✓ Excellent for network analysis (transport, utilities, routing)
- ✓ Supports topology-based operations (adjacency, connectivity)

---

### Summary

- **Raster = “Fields & Surfaces”** (continuous phenomena)
- **Vector = “Objects & Boundaries”** (discrete features)

Most real-world GIS applications combine both for complete spatial understanding.

In [ ]:
!pip install fiona >> /dev/null

In [ ]:
# CORE NUMERIC / DATA STACK
import numpy as np
import pandas as pd

# GEOSPATIAL (RASTER)
import rasterio
import rasterio.plot
import rasterio.merge
import rasterio.mask

from osgeo import gdal

# GEOSPATIAL (VECTOR)
import geopandas as gpd
import fiona
import osmnx as ox

# Coordinate reference systems / projections
import pyproj

# Geometry operations
from shapely.geometry import Polygon

# VISUALIZATION
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Create a simple raster (grid of values)
# Temperature map of a 4x4 km area with 1 km resolution
temperature_raster = np.array(
    [
        [15, 16, 18, 20],  # Top row (Y=3) when visual orientation is normal
        [15, 17, 19, 22],  # Row (Y=2)
        [16, 18, 20, 24],  # Row (Y=1)
        [17, 19, 21, 25],  # Bottom row (Y=0)
    ]
)

# reate vector data: administrative boundaries (irregular polygons)
# These represent 4 city zones within the same 4x4 km area
zone_polygons = [
    Polygon([(0, 2), (2, 2), (2, 4), (0, 4)]),  # NW zone
    Polygon([(2, 2), (4, 2), (4, 4), (2, 4)]),  # NE zone
    Polygon([(0, 0), (2, 0), (2, 2), (0, 2)]),  # SW zone
    Polygon([(2, 0), (4, 0), (4, 2), (2, 2)]),  # SE zone
]
zone_names = ["Northwest", "Northeast", "Southwest", "Southeast"]

# Visualization Setup
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ["red", "blue", "green", "orange"]

# --- Plot 1: Raster data ---
# We omit origin='lower' to keep matrix indexing matching spatial coordinates cleanly
im = axes[0].imshow(
    temperature_raster, cmap="RdYlBu_r", extent=[0, 4, 0, 4]
)
axes[0].set_title(
    "RASTER DATA\n(Temperature grid)", fontsize=12, fontweight="bold"
)
axes[0].set_xlabel("Longitude (km)")
axes[0].set_ylabel("Latitude (km)")
axes[0].grid(True, alpha=0.3)

# Add pixel grid labels dynamically based on background brightness
for i in range(4):
    for j in range(4):
        val = temperature_raster[i, j]
        # Dynamically change text color based on temperature value for readability
        text_color = "white" if val > 18 else "black"
        # Calculate standard center grid coordinate: Y is inverted relative to matrix row index
        axes[0].text(
            j + 0.5,
            3.5 - i,
            f"{val}°C",
            ha="center",
            va="center",
            fontsize=10,
            color=text_color,
            fontweight="bold",
        )
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# --- Plot 2: Vector data ---
axes[1].set_xlim(0, 4)
axes[1].set_ylim(0, 4)
for poly, name, color in zip(zone_polygons, zone_names, colors):
    x, y = poly.exterior.xy
    axes[1].fill(x, y, alpha=0.3, edgecolor=color, linewidth=2, label=name)
axes[1].set_title(
    "VECTOR DATA\n(Zone boundaries)", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Longitude (km)")
axes[1].set_ylabel("Latitude (km)")
axes[1].legend(loc="upper left", fontsize=9)
axes[1].grid(True, alpha=0.3)

# --- Plot 3: Overlay comparison ---
axes[2].imshow(
    temperature_raster, cmap="RdYlBu_r", extent=[0, 4, 0, 4], alpha=0.7
)
for poly, color in zip(zone_polygons, colors):
    x, y = poly.exterior.xy
    axes[2].plot(x, y, color=color, linewidth=2)
axes[2].set_title("Overlay: Raster + Vector", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Longitude (km)")
axes[2].set_ylabel("Latitude (km)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Console Printout Details
print(
    f"Raster: {temperature_raster.shape[0]} x {temperature_raster.shape[1]} grid = {temperature_raster.size} pixels"
)
print(
    f"Vector: {len(zone_polygons)} polygons, each with variable number of vertices"
)

**What is a raster?** A raster is a type of digital image made up of pixels, where the color of each individual pixel is stored separately. On the other hand, vector images use mathematical formulas to represent the shapes in the image instead of storing each pixel color separately. In GIS applications, remote sensing data and maps are often stored in raster images.

**In this notebook, rasterio package will be used to load and process raster data**

## Getting topological raster data
We're going to use a Digital Elevation Model from USGS. 

You can search for data products from USGS [here](https://apps.nationalmap.gov/downloader/). We are using elevation products (3DEP) at 1/3 arc-second (~10m) resolution. Each tile spans 1x1 degree.

These files are quite large - it will take a while to download (approximately 1 min).

In [ ]:
# Get raster of boston-area elevations (takes a while)
tile1_url = 'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/current/n43w072/USGS_13_n43w072.tif'
tile2_url = 'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/current/n43w071/USGS_13_n43w071.tif'

# write them to local files so we don't have to keep downloading
local_tile1_location = 'output/local_tile1.tif'
local_tile2_location = 'output/local_tile2.tif'

# Create output directory if it doesn't exist
!mkdir -p output

def download_to_local(tile_url, local_tile_path):
    '''
    Download a raster file from URL and save locally.
    Converts to float64 for decimal elevation values.
    '''
    with rasterio.Env():
        with rasterio.open(tile_url) as src:
            profile = src.profile
            profile['dtype'] = 'float64'
            profile['indexes'] = 1
            print(profile)
            with rasterio.open(local_tile_path, 'w', **profile) as dst:
                dst.write_band(1,src.read(1).squeeze().astype(np.float64))

# download both tiles
download_to_local(tile1_url, local_tile1_location)
print("*********************")
download_to_local(tile2_url, local_tile2_location)

In [ ]:
# Large rasters are slow to process. We'll downsample to reduce file size 
# without losing important elevation patterns. E.g., rescaling_factor=0.25 
# shrinks the raster 4× in each direction (16× smaller overall).

def resize_tile(tile_location, rescaling_factor):
    """Downsample raster to lower resolution for faster processing."""
    with rasterio.open(tile_location) as src:
        print(f'Original: {src.width}×{src.height}')
        
        new_height = int(src.height * rescaling_factor) # int() to round to nearest pixel
        new_width = int(src.width * rescaling_factor)
        
        # Bilinear: blend pixels smoothly (good for elevation), out_shape: dimension to be resampled to
        tile = src.read(1, out_shape=(new_height, new_width), 
                       resampling=rasterio.enums.Resampling.bilinear)

        # copy metadata
        tile_profile = src.profile.copy()
        
        # Update georeference (transform, height, width) for smaller pixels
        scale_x = src.width / tile.shape[-1]
        scale_y = src.height / tile.shape[-2]
        tile_profile['transform'] = src.transform * src.transform.scale(scale_x, scale_y)
        tile_profile['height'] = new_height
        tile_profile['width'] = new_width
        
        print(f'Downsampled: {new_width}×{new_height}')
        
        return tile, tile_profile

In [ ]:
rescaling_factor = 0.25 # cut them down to 1/4 in each direction (height and width) resulting in 16x smaller profile

print("Tile 1:")
tile1, tile1_profile = resize_tile(local_tile1_location, rescaling_factor)

print("Tile 2:")
tile2, tile2_profile = resize_tile(local_tile2_location, rescaling_factor)

**Now let's also save the low resolution versions to file for ease of reference. We'll use the GeoTiff format**

In [ ]:
tile1_profile['driver'] = 'GTiff'
tile2_profile['driver'] = 'GTiff'

lowres_tile1_loc = 'lowres_tile1.tif'
lowres_tile2_loc = 'lowres_tile2.tif'

with rasterio.open(lowres_tile1_loc, 'w', **tile1_profile) as outfile:
    outfile.write(tile1, 1)
    
with rasterio.open(lowres_tile2_loc, 'w', **tile2_profile) as outfile:
    outfile.write(tile2, 1)

In [ ]:
print("Tile 1: ", tile1.shape, "Tile 2:", tile2.shape)

**Now let's visualize**

In [ ]:
# Visualize both tiles side by side
fig = plt.figure(figsize=(14, 5))

# Left: Tile 1 (Boston West)
ax1 = fig.add_subplot(1, 2, 1)
rasterio.plot.show(tile1, ax=ax1, transform=tile1_profile['transform'])
ax1.set_title('Tile 1: Boston West')

# Right: Tile 2 (Boston East)
ax2 = fig.add_subplot(1, 2, 2)
rasterio.plot.show(tile2, ax=ax2, transform=tile2_profile['transform'])
ax2.set_title('Tile 2: Boston East')

plt.tight_layout()
plt.show()

**NOTE:** Tile 2 looks weird because it contains invalid data marked as `nodata`=`-3.4e+38`. These represent areas with no elevation (water, clouds, bad readings). We need to mask them out so they don't break the visualization.

Fortunately, there's a masking function that lets us select all of the non-nodata pixelss

In [ ]:
# Read masks: identify which pixels are valid (1) vs nodata (0)
with rasterio.open(lowres_tile1_loc) as src:
    tile1_mask = src.read_masks(1)
    
with rasterio.open(lowres_tile2_loc) as src:
    tile2_mask = src.read_masks(1)

In [ ]:
print(tile2_mask)

In [ ]:
# we use the np.ma.masked_where() function to mask out all the values equal to zero

# Visualize, masking out nodata pixels so they don't appear
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 2, 1)
rasterio.plot.show(np.ma.masked_where(tile1_mask==0, tile1), ax=ax1, transform=tile1_profile['transform'])
ax1.set_title('Tile 1: Boston West')

ax2 = fig.add_subplot(1, 2, 2)
rasterio.plot.show(np.ma.masked_where(tile2_mask==0, tile2), ax=ax2, transform=tile2_profile['transform'])
ax2.set_title('Tile 2: Boston East')

plt.tight_layout()
plt.show()

Let's combine the two tiles together into one large mosaic and save it as a geotiff

In [ ]:
combined_DEM = 'combinedDEM.tif'

In [ ]:
with rasterio.open(lowres_tile1_loc, 'r') as src1:
    with rasterio.open(lowres_tile2_loc, 'r') as src2:
        combined, out_transform = rasterio.merge.merge([src1, src2],
                                                       nodata=src2.profile['nodata'])
        combined_meta = src1.profile.copy()
        combined_meta.update({'driver':'GTiff',
                         'count': combined.shape[0],
                        'height': combined.shape[1],
                        'width': combined.shape[2],
                        'transform': out_transform,
                        'crs': src1.crs})
    with rasterio.open(combined_DEM, 'w', **combined_meta) as dst:
        dst.write(combined)

In [ ]:
with rasterio.open(combined_DEM, 'r') as src:
    print(src.meta)
    combined = src.read(1)
    combined_mask = src.read_masks(1)
    combined_transform = src.profile['transform']
    rasterio.plot.show(np.ma.masked_where(combined_mask==0, combined), 
                     transform=combined_transform)

Let's take a look at what cambridge and boston look like on this map

In [ ]:
# Geocode city boundaries (convert city names to geographic polygons)
cambridge_name = "Cambridge, MA, USA"
graph_area_cambridge = ox.geocode_to_gdf(cambridge_name)

boston_name = "Boston, MA, USA"
graph_area_boston = ox.geocode_to_gdf(boston_name)

# Combine both city boundaries into one GeoDataFrame
local_areas = pd.concat([graph_area_boston, graph_area_cambridge], ignore_index=True)

!mkdir -p local_areas

# Convert to EPSG:4979 (3D lat/lon) and save as GeoJSON
local_areas.to_crs(epsg=4979).to_file('local_areas/local_areas.geojson', driver='GeoJSON')

In [ ]:
local_areas.head()

In [ ]:
# Create figure and axis for plotting
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)

# Plot the merged elevation raster (masked to hide nodata)
p = rasterio.plot.show(np.ma.masked_where(combined_mask==0, combined), 
                       ax=ax, transform=combined_transform)

# Overlay city boundaries as red and magenta polygons
# Use EPSG:4979 to match the raster CRS
graph_area_cambridge.to_crs(epsg=4979).plot(ax=ax, facecolor='none', edgecolor='red', label='Cambridge')
graph_area_boston.to_crs(epsg=4979).plot(ax=ax, facecolor='none', edgecolor='magenta', label='Boston')

plt.title('Elevation with City Boundaries')
plt.show()

We can [crop the raster using these vectors](https://rasterio.readthedocs.io/en/latest/topics/masking-by-shapefile.html)

In [ ]:
# Use vector boundaries to crop the raster (mask.mask)
with rasterio.open(combined_DEM, 'r') as src:
    # Crop elevation raster to match city boundaries
    # mask() returns: cropped data and new georeference transform
    out_image, out_transform = rasterio.mask.mask(src, local_areas['geometry'].values, crop=True)
    
    # Copy metadata from original raster
    out_profile = src.profile
    
    # Update dimensions to match cropped data
    out_profile.update({"driver": "GTiff",
                 "height": out_image.shape[1],
                 "width": out_image.shape[2],
                 "transform": out_transform})

# Save the masked/cropped raster to file
with rasterio.open("masked_elevation_local.tif", "w", **out_profile) as masked_outfile:
    masked_outfile.write(out_image)

In [ ]:
# Load and display the final masked elevation raster
with rasterio.open('masked_elevation_local.tif', 'r') as src:
    print(src.meta)  # Show file metadata
    
    local_crop = src.read(1)  # Read elevation data
    local_crop_mask = src.read_masks(1)  # Get valid pixel locations
    local_transform = src.profile['transform']  # Get georeference
    
    # Plot with masking to hide invalid pixels
    rasterio.plot.show(np.ma.masked_where(local_crop_mask==0, local_crop), 
                     transform=local_transform)

# GDAL
Another package that's very useful for processing raster is GDAL, which is used behind the scenes for a lot of GIS applications. https://gdal.org/index.html

In this example, we'll be using it to create a hillshade. GDAL is available as a set of command line tools, which are installed in `/opt/conda/bin/`
In particular, we're using `gdaldem` which is the gdal tool for working with digital elevation models. For more info about the capabilities of gdaldem, check the [documentation](https://gdal.org/programs/gdaldem.html#gdaldem)

In [ ]:
%%bash
mamba create -n gdal gdal
mamba init
source ~/.bashrc

!mamba activate gdal

In [ ]:
!mamba run -n gdal gdaldem hillshade combinedDEM.tif hs.tif

In [ ]:
with rasterio.open('hs.tif', 'r') as src:
    print(src.meta)
    hillshade = src.read(1)
    hillshade_mask = src.read_masks(1)
    hillshade_transform = src.profile['transform']
    rasterio.plot.show(np.ma.masked_where(hillshade_mask==0, hillshade), 
                     transform=hillshade_transform)

There are also python bindings for the GDAL functions.
- [Example scripts](https://github.com/OSGeo/gdal/tree/master/swig/python/gdal-utils/osgeo_utils/samples)
- [API reference](https://gdal.org/python/index.html)


Check out the other GDAL command line tools: https://gdal.org/programs/index.html